# Logistic Regression: From Math Equations to PyTorch Code

**Course:** Deep Learning  
**Lab length:** ~2 hours  
**Reading:** Chapter 1 of *Hundred-Page Language Model Book*  
**Focus:** Connecting the mathematical foundations of logistic regression to PyTorch implementation

### Learning Goals
By the end of this lab, you will be able to:
1. **Understand the mathematical foundation** of logistic regression from Chapter 1
2. **Connect equations to PyTorch code** step-by-step
3. **Visualize the learning process** and decision boundaries
4. **Implement logistic regression using `nn.Sequential`** 
5. **Trace through forward/backward passes** connecting math to code

### Key Mathematical Concepts (from Chapter 1)
- **Linear transformation:** $z = \mathbf{w}^\top \mathbf{x} + b$
- **Activation function:** $\hat{y} = \sigma(z) = \frac{1}{1 + e^{-z}}$
- **Loss function:** $\mathcal{L} = -\frac{1}{n}\sum_{i=1}^n [y_i \log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i)]$
- **Gradient descent:** $\mathbf{w} \leftarrow \mathbf{w} - \eta \nabla_{\mathbf{w}}\mathcal{L}$

## 0) Setup and Imports

Let's import the necessary libraries and set up our environment.

In [ ]:
import torch
from torch import nn
import matplotlib.pyplot as plt
import numpy as np

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check if GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Set plotting style for better visualizations
plt.style.use('default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 1) Mathematical Foundation (Chapter 1)

Before we dive into code, let's review the key equations from Chapter 1:

### The Logistic Regression Model

**Step 1: Linear Transformation**
$$z = \mathbf{w}^\top \mathbf{x} + b$$

**Step 2: Activation Function**
$$\hat{y} = \sigma(z) = \frac{1}{1 + e^{-z}}$$

**Step 3: Loss Function (Binary Cross-Entropy)**
$$\mathcal{L} = -\frac{1}{n}\sum_{i=1}^n [y_i \log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i)]$$

**Step 4: Gradient Descent Update**
$$\mathbf{w} \leftarrow \mathbf{w} - \eta \nabla_{\mathbf{w}}\mathcal{L}$$
$$b \leftarrow b - \eta \nabla_b\mathcal{L}$$

### Visual Representation
```
Input x → [Linear: w^T x + b] → [Sigmoid: σ(z)] → Output ŷ
```

This is exactly what we'll implement in PyTorch!

## 2) Create a Simple 2D Binary Dataset

We'll generate two Gaussian blobs in 2D so we can **see** the decision boundary evolve during training.

- **Class 0** (blue): centered near (-2, -2)
- **Class 1** (red): centered near (+2, +2)

In [ ]:
# Generate synthetic data
n_per_class = 300
dim = 2

# Class centers
mean0 = torch.tensor([-2.0, -2.0])
mean1 = torch.tensor([ 2.0,  2.0])

# Shared covariance matrix
cov = torch.tensor([[1.0, 0.3],
                    [0.3, 1.0]])

# Generate samples using Cholesky decomposition
L = torch.linalg.cholesky(cov)
z0 = torch.randn(n_per_class, dim) @ L.T + mean0
z1 = torch.randn(n_per_class, dim) @ L.T + mean1

# Combine and shuffle
X = torch.cat([z0, z1], dim=0)
y = torch.cat([torch.zeros(n_per_class), torch.ones(n_per_class)], dim=0).unsqueeze(1)

perm = torch.randperm(X.size(0))
X, y = X[perm], y[perm]

# Move to device
X, y = X.to(device), y.to(device)

print(f"Dataset shape: {X.shape}")
print(f"Class balance: {y.float().mean().item():.3f} (should be ~0.5)")

## 3) Visualization Functions

We'll create enhanced visualization functions to show:
- Data points with class colors
- Decision boundaries
- Training progress
- Loss curves

In [ ]:
def plot_data_with_boundary(X, y, w=None, b=None, title=None, show_decision=True):
    """Enhanced plotting function with better visuals"""
    X_cpu = X.detach().cpu()
    y_cpu = y.detach().cpu().squeeze()

    plt.figure(figsize=(8, 6))
    
    # Plot data points with better styling
    plt.scatter(X_cpu[y_cpu==0, 0], X_cpu[y_cpu==0, 1], 
                s=50, alpha=0.7, label="Class 0", color='blue', edgecolors='white', linewidth=0.5)
    plt.scatter(X_cpu[y_cpu==1, 0], X_cpu[y_cpu==1, 1], 
                s=50, alpha=0.7, label="Class 1", color='red', edgecolors='white', linewidth=0.5)

    # Plot decision boundary if weights are provided
    if w is not None and b is not None and show_decision:
        w = w.detach().cpu().view(-1)
        b = b.detach().cpu().view(())
        
        if abs(w[1].item()) > 1e-8:
            x1_vals = torch.linspace(X_cpu[:,0].min()-1, X_cpu[:,0].max()+1, 200)
            x2_vals = -(w[0]/w[1]) * x1_vals - b/w[1]
            x2_vals = x2_vals.clamp(X_cpu[:,1].min()-2, X_cpu[:,1].max()+2)
            
            plt.plot(x1_vals, x2_vals, 'k-', linewidth=3, alpha=0.8, label='Decision Boundary')
            
            # Add arrow showing positive side
            mid_idx = len(x1_vals) // 2
            arrow_x, arrow_y = x1_vals[mid_idx], x2_vals[mid_idx]
            plt.annotate('ŷ > 0.5', xy=(arrow_x, arrow_y), 
                        xytext=(arrow_x + 1, arrow_y + 1),
                        arrowprops=dict(arrowstyle='->', lw=2, color='green'),
                        fontsize=12, color='green')

    plt.legend(fontsize=12)
    if title:
        plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel("Feature 1 (x₁)", fontsize=12)
    plt.ylabel("Feature 2 (x₂)", fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_training_progress(losses, accuracies=None, title="Training Progress"):
    """Plot training curves"""
    fig, axes = plt.subplots(1, 2 if accuracies else 1, figsize=(12, 4))
    
    if accuracies:
        # Loss subplot
        axes[0].plot(losses, 'b-', linewidth=2, alpha=0.8)
        axes[0].set_title('Loss vs. Epoch', fontweight='bold')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Loss')
        axes[0].set_ylim(0, max(losses) * 1.1)
        axes[0].grid(True, alpha=0.3)
        
        # Accuracy subplot
        axes[1].plot(accuracies, 'r-', linewidth=2, alpha=0.8)
        axes[1].set_title('Accuracy vs. Epoch', fontweight='bold')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Accuracy')
        axes[1].set_ylim(0, 1)
        axes[1].grid(True, alpha=0.3)
    else:
        axes.plot(losses, 'b-', linewidth=2, alpha=0.8)
        axes.set_title('Loss vs. Epoch', fontweight='bold')
        axes.set_xlabel('Epoch')
        axes.set_ylabel('Loss')
        axes.set_ylim(0, max(losses) * 1.1)
        axes.grid(True, alpha=0.3)
    
    plt.suptitle(title, fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

def plot_equation_mapping():
    """Visual mapping from equations to PyTorch code"""
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    
    # Create a visual diagram
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.axis('off')
    
    # Math equations
    ax.text(1, 8, 'Math Equations:', fontsize=16, fontweight='bold', color='blue')
    ax.text(1, 7, '$z = \mathbf{w}^\top \mathbf{x} + b$', fontsize=14, color='blue')
    ax.text(1, 6, '$\hat{y} = \sigma(z) = \frac{1}{1 + e^{-z}}$', fontsize=14, color='blue')
    ax.text(1, 5, '$\mathcal{L} = -\frac{1}{n}\sum_{i=1}^n [y_i \log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i)]$', fontsize=12, color='blue')
    
    # PyTorch code
    ax.text(6, 8, 'PyTorch Implementation:', fontsize=16, fontweight='bold', color='red')
    ax.text(6, 7, 'nn.Linear(2, 1)', fontsize=14, color='red')
    ax.text(6, 6, 'nn.Sigmoid()', fontsize=14, color='red')
    ax.text(6, 5, 'nn.BCELoss()', fontsize=14, color='red')
    
    # Arrows connecting them
    ax.arrow(4.5, 7, 1, 0, head_width=0.2, head_length=0.2, fc='green', ec='green', linewidth=2)
    ax.arrow(4.5, 6, 1, 0, head_width=0.2, head_length=0.2, fc='green', ec='green', linewidth=2)
    ax.arrow(4.5, 5, 1, 0, head_width=0.2, head_length=0.2, fc='green', ec='green', linewidth=2)
    
    # Final model structure
    ax.text(3, 3, 'Complete Model:', fontsize=16, fontweight='bold', color='purple')
    ax.text(3, 2, 'nn.Sequential(', fontsize=12, color='purple')
    ax.text(4, 1.5, 'nn.Linear(2, 1),', fontsize=12, color='purple')
    ax.text(4, 1, 'nn.Sigmoid()', fontsize=12, color='purple')
    ax.text(3, 0.5, ')', fontsize=12, color='purple')
    
    plt.title('Math Equations ↔ PyTorch Code Mapping', fontsize=18, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot our dataset
plot_data_with_boundary(X, y, title="Our Binary Classification Dataset")

# Show the equation mapping
plot_equation_mapping()

## 4) PyTorch Implementation: Connecting Math to Code

Now let's implement logistic regression using PyTorch's `nn.Sequential`. We'll trace through each step to see how the math equations map to code.

### Step-by-Step Breakdown

**1. Linear Transformation:** $z = \mathbf{w}^\top \mathbf{x} + b$  
→ `nn.Linear(2, 1)` implements exactly this!

**2. Sigmoid Activation:** $\hat{y} = \sigma(z) = \frac{1}{1 + e^{-z}}$  
→ `nn.Sigmoid()` implements exactly this!

**3. Loss Function:** $\mathcal{L} = -\frac{1}{n}\sum_{i=1}^n [y_i \log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i)]$  
→ `nn.BCELoss()` implements exactly this!

**4. Optimization:** $\mathbf{w} \leftarrow \mathbf{w} - \eta \nabla_{\mathbf{w}}\mathcal{L}$  
→ `torch.optim.SGD` implements exactly this!

In [ ]:
# Create our logistic regression model
model = nn.Sequential(
    nn.Linear(2, 1),    # z = w^T x + b
    nn.Sigmoid()         # ŷ = σ(z)
).to(device)

print("Model architecture:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters())}")
print(f"\nInitial weights: {model[0].weight.data.flatten().tolist()}")
print(f"Initial bias: {model[0].bias.data.item():.4f}")

### Training Setup

We'll use:
- **Loss function:** `nn.BCELoss()` (Binary Cross-Entropy)
- **Optimizer:** `torch.optim.SGD` (Stochastic Gradient Descent)
- **Learning rate:** $\eta = 0.5$

The training loop will:
1. Forward pass: compute $\hat{y} = \sigma(\mathbf{w}^\top \mathbf{x} + b)$
2. Compute loss: $\mathcal{L} = \text{BCE}(\hat{y}, y)$
3. Backward pass: compute gradients $\nabla_{\mathbf{w}}\mathcal{L}$ and $\nabla_b\mathcal{L}$
4. Update parameters: $\mathbf{w} \leftarrow \mathbf{w} - \eta \nabla_{\mathbf{w}}\mathcal{L}$

In [ ]:
# Training setup
criterion = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

# Training loop
epochs = 100
losses = []
accuracies = []

print("Training started...")
print("Epoch | Loss    | Accuracy | Decision Boundary")
print("-" * 50)

for epoch in range(epochs):
    # Forward pass: ŷ = σ(w^T x + b)
    y_pred = model(X)
    
    # Compute loss: L = BCE(ŷ, y)
    loss = criterion(y_pred, y)
    
    # Backward pass: compute gradients
    optimizer.zero_grad()
    loss.backward()
    
    # Update parameters: w ← w - η∇wL
    optimizer.step()
    
    # Compute accuracy
    with torch.no_grad():
        predictions = (y_pred >= 0.5).float()
        accuracy = (predictions == y).float().mean().item()
    
    losses.append(loss.item())
    accuracies.append(accuracy)
    
    # Print progress every 20 epochs
    if (epoch + 1) % 20 == 0 or epoch == 0:
        # Extract current weights for visualization
        with torch.no_grad():
            w_current = model[0].weight.data.clone().T
            b_current = model[0].bias.data.clone().view(1)
        
        print(f"{epoch+1:5d} | {loss.item():.4f} | {accuracy:.4f} | w=[{w_current[0,0]:.3f}, {w_current[0,1]:.3f}], b={b_current.item():.3f}")
        
        # Show decision boundary evolution
        plot_data_with_boundary(X, y, w_current, b_current, 
                               title=f"Decision Boundary at Epoch {epoch+1}")

print("\nTraining completed!")
print(f"Final accuracy: {accuracies[-1]:.4f}")
print(f"Final loss: {losses[-1]:.4f}")

## 5) Visualizing the Learning Process

Let's examine what happened during training:

1. **Loss curve:** How quickly did the model learn?
2. **Accuracy curve:** How well did it classify the data?
3. **Decision boundary evolution:** How did the separating line move?

In [ ]:
# Plot training progress
plot_training_progress(losses, accuracies, "Logistic Regression Training Progress")

# Show final decision boundary
with torch.no_grad():
    w_final = model[0].weight.data.clone().T
    b_final = model[0].bias.data.clone().view(1)

plot_data_with_boundary(X, y, w_final, b_final, 
                       title="Final Decision Boundary (Trained Model)")

print(f"Final weights: w = [{w_final[0,0]:.4f}, {w_final[0,1]:.4f}]")
print(f"Final bias: b = {b_final.item():.4f}")
print(f"\nThe decision boundary equation is:")
print(f"{w_final[0,0]:.4f} × x₁ + {w_final[0,1]:.4f} × x₂ + {b_final.item():.4f} = 0")

## 6) Math ↔ Code Connection Summary

Let's summarize how each mathematical concept maps to PyTorch code:

| Mathematical Concept | Equation | PyTorch Implementation |
|---------------------|----------|------------------------|
| **Linear Transformation** | $z = \mathbf{w}^\top \mathbf{x} + b$ | `nn.Linear(2, 1)` |
| **Activation Function** | $\hat{y} = \sigma(z) = \frac{1}{1 + e^{-z}}$ | `nn.Sigmoid()` |
| **Loss Function** | $\mathcal{L} = -\frac{1}{n}\sum_{i=1}^n [y_i \log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i)]$ | `nn.BCELoss()` |
| **Parameter Update** | $\mathbf{w} \leftarrow \mathbf{w} - \eta \nabla_{\mathbf{w}}\mathcal{L}$ | `optimizer.step()` |

### Key Insight
**Logistic regression is a one-neuron neural network!** The `nn.Sequential` approach shows how we can stack multiple layers to create more complex models. This is the foundation of deep learning.

## 7) Exercises (Solutions Included)

> **Note:** These are solutions for the instructor. Remove the solution code blocks before giving to students.

### Exercise 1: Feature Scaling Effect
**Question:** What happens if we standardize the features (mean=0, std=1) before training? How does this affect convergence and the final decision boundary?

**Solution:** Standardization often improves convergence speed and numerical stability.

In [ ]:
# === SOLUTION: Feature Standardization ===
with torch.no_grad():
    # Standardize features
    mu = X.mean(dim=0, keepdim=True)
    sigma = X.std(dim=0, unbiased=False, keepdim=True).clamp_min(1e-6)
X_std = (X - mu) / sigma

# Create and train model on standardized data
model_std = nn.Sequential(nn.Linear(2,1), nn.Sigmoid()).to(device)
opt_std = torch.optim.SGD(model_std.parameters(), lr=0.5)
crit = nn.BCELoss()

losses_std = []
for t in range(80):
    opt_std.zero_grad()
    loss = crit(model_std(X_std), y)
    loss.backward()
    opt_std.step()
    losses_std.append(loss.item())

# Map weights back to original scale for comparison
with torch.no_grad():
    W_std = model_std[0].weight.data.clone().T / sigma.T
    b_std = model_std[0].bias.data.clone().view(1) - (mu @ (model_std[0].weight.data.clone().T / sigma.T)).view(1)

print("Standardized training results:")
print(f"Final loss: {losses_std[-1]:.4f}")
print(f"Weights: [{W_std[0,0]:.4f}, {W_std[0,1]:.4f}]")
print(f"Bias: {b_std.item():.4f}")

plot_data_with_boundary(X, y, W_std, b_std, title="Standardized Training (Mapped Back to Original Scale)")

### Exercise 2: Learning Rate Sensitivity
**Question:** How does the learning rate affect training? Try different values and observe the impact on convergence and decision boundaries.

**Solution:** Learning rate controls step size in gradient descent.

In [ ]:
# === SOLUTION: Learning Rate Comparison ===
def train_with_lr(lr=0.5, epochs=80):
    """Train model with specified learning rate"""
    m = nn.Sequential(nn.Linear(2,1), nn.Sigmoid()).to(device)
    opt = torch.optim.SGD(m.parameters(), lr=lr)
    crit = nn.BCELoss()
    losses = []
    
    for t in range(epochs):
        opt.zero_grad()
        yhat = m(X)
        loss = crit(yhat, y)
        loss.backward()
        opt.step()
        losses.append(float(loss.detach().cpu()))
    
    with torch.no_grad():
        W = m[0].weight.data.clone().T
        b = m[0].bias.data.clone().view(1)
    return losses, W, b

# Train with different learning rates
losses_005, W005, b005 = train_with_lr(lr=0.05, epochs=120)
losses_10,  W10,  b10  = train_with_lr(lr=1.0, epochs=120)

# Plot comparison
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(losses_005, 'b-', linewidth=2, label="lr=0.05")
plt.plot(losses_10, 'r-', linewidth=2, label="lr=1.0")
plt.title('Loss vs. Epoch (Different Learning Rates)', fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 2)
plot_data_with_boundary(X, y, W005, b005, title="Boundary: lr=0.05")

plt.subplot(1, 3, 3)
plot_data_with_boundary(X, y, W10, b10, title="Boundary: lr=1.0")

plt.tight_layout()
plt.show()

print("Learning rate comparison:")
print(f"lr=0.05: Final loss = {losses_005[-1]:.4f}")
print(f"lr=1.0:  Final loss = {losses_10[-1]:.4f}")

### Exercise 3: Numerical Stability
**Question:** Why do we use `nn.BCELoss()` with `nn.Sigmoid()` instead of `nn.BCEWithLogitsLoss()`? What are the trade-offs?

**Solution:** `BCEWithLogitsLoss` combines sigmoid and BCE for numerical stability.

In [ ]:
# === SOLUTION: BCEWithLogitsLoss Comparison ===
model_logit = nn.Sequential(
    nn.Linear(2,1)  # No Sigmoid - outputs logits
).to(device)

crit_logit = nn.BCEWithLogitsLoss()  # Combines sigmoid + BCE
opt_logit = torch.optim.SGD(model_logit.parameters(), lr=0.5)

losses_logit = []
for t in range(100):
    opt_logit.zero_grad()
    logits = model_logit(X)         # Raw scores (logits)
    loss = crit_logit(logits, y)    # Applies sigmoid internally
    loss.backward()
    opt_logit.step()
    losses_logit.append(loss.item())

with torch.no_grad():
    W_log = model_logit[0].weight.data.clone().T
    b_log = model_logit[0].bias.data.clone().view(1)

print("BCEWithLogitsLoss results:")
print(f"Final loss: {losses_logit[-1]:.4f}")
print(f"Weights: [{W_log[0,0]:.4f}, {W_log[0,1]:.4f}]")
print(f"Bias: {b_log.item():.4f}")

plot_data_with_boundary(X, y, W_log, b_log, title="BCEWithLogitsLoss (No Sigmoid in Model)")

print("\nKey differences:")
print("1. BCEWithLogitsLoss is more numerically stable")
print("2. Avoids potential overflow in sigmoid computation")
print("3. Often converges faster and more reliably")

## 8) Reflection and Next Steps

### What We've Accomplished
✅ **Connected math equations to PyTorch code** - Each mathematical concept has a direct PyTorch implementation  
✅ **Visualized the learning process** - Saw decision boundaries evolve during training  
✅ **Implemented logistic regression** using `nn.Sequential`  
✅ **Understood the training loop** - Forward pass, loss computation, backward pass, parameter updates

### Key Insights
1. **Logistic regression is a neural network** - It's a single neuron with sigmoid activation
2. **Math maps directly to code** - Each equation has a corresponding PyTorch component
3. **Visualization is powerful** - Seeing decision boundaries evolve helps understand learning
4. **PyTorch handles the complexity** - We focus on the model architecture, PyTorch handles gradients

### Next Steps in Deep Learning
This foundation prepares you for:
- **Multi-layer networks** - Stack more `nn.Linear` + activation layers
- **Different architectures** - CNNs, RNNs, Transformers
- **Advanced optimization** - Adam, learning rate scheduling
- **Regularization** - Dropout, weight decay, batch normalization

### Questions for Discussion
- Which representation (math vs. code) made the connection clearest for you?
- Where does the **nonlinearity** come from in logistic regression?
- What would happen if we removed the sigmoid activation?
- How does this relate to the neural network concepts in Chapter 1?